# Submission 2 - 234-class multi-label inference

Forked from `birdclef-2026-submission-1.ipynb`. Uses the multi-label model trained in `birdclef_plus_2026_multilabel_attempt.ipynb`, so all 234 columns (including the 28 species missing from `train_audio`) get real probabilities instead of being zero-filled.

In [ ]:
import json
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from fastai.vision.all import load_learner, PILImage

In [ ]:
import kagglehub
path = kagglehub.competition_download('birdclef-2026')
print('Path to competition files:', path)

In [ ]:
MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/1/model_multilabel_234.pkl'
VOCAB_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/1/vocab.json'
TEST_DIR = Path(path) / 'test_soundscapes'
TARGET_SIZE = (224, 224)
CLIP_DURATION = 5
SAMPLE_RATE = 32000
BATCH_SIZE = 64

In [ ]:
sample_sub = pd.read_csv(Path(path) / 'sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']
assert len(all_species) == 234, f'expected 234, got {len(all_species)}'

learn = load_learner(MODEL_PATH)
model_vocab = list(learn.dls.vocab)

if Path(VOCAB_PATH).exists():
    saved_vocab = json.load(open(VOCAB_PATH))
    assert saved_vocab == model_vocab, 'vocab.json disagrees with learner.dls.vocab'

assert model_vocab == all_species, (
    'Model vocab does not match sample_submission column order. '
    f'len(model_vocab)={len(model_vocab)}, len(all_species)={len(all_species)}; '
    f'first mismatch at {next((i for i, (a, b) in enumerate(zip(model_vocab, all_species)) if a != b), None)}'
)
print('Vocab order matches sample_submission for all 234 classes.')

In [ ]:
def audio_to_pil(samples: np.ndarray) -> Image.Image:
    s = librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE)
    s_db = librosa.power_to_db(s, ref=np.max)
    s_min, s_max = float(s_db.min()), float(s_db.max())
    if s_max == s_min:
        s_norm = np.zeros_like(s_db, dtype=np.uint8)
    else:
        s_norm = ((s_db - s_min) / (s_max - s_min) * 255).astype(np.uint8)
    return Image.fromarray(s_norm).resize(TARGET_SIZE).convert('RGB')

test_files = sorted(TEST_DIR.glob('*.ogg'))
print(f'Found {len(test_files)} test soundscape files')

row_ids = []
imgs = []
clip_length = CLIP_DURATION * SAMPLE_RATE

for soundscape in test_files:
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    start = 0
    end_time = CLIP_DURATION
    while start + clip_length <= len(samples):
        chunk = samples[start:start + clip_length]
        imgs.append(PILImage(audio_to_pil(chunk)))
        row_ids.append(f'{soundscape.stem}_{end_time}')
        start += clip_length
        end_time += CLIP_DURATION

print(f'Built {len(imgs)} 5s windows for inference.')

In [ ]:
test_dl = learn.dls.test_dl(imgs, bs=BATCH_SIZE)
preds, _ = learn.get_preds(dl=test_dl)
preds_np = preds.cpu().numpy()
assert preds_np.shape == (len(row_ids), 234), preds_np.shape
print('Predictions shape:', preds_np.shape, '  range:', float(preds_np.min()), '..', float(preds_np.max()))

In [ ]:
submission = pd.DataFrame(preds_np, columns=all_species)
submission.insert(0, 'row_id', row_ids)
assert list(submission.columns) == ['row_id'] + all_species
submission.to_csv('submission.csv', index=False)
print(f'Done. {len(submission)} rows written.')
submission.head()